In [ ]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Download stopwords
nltk.download('stopwords')

# --------------------------------------------------
# 1. Load Dataset
# --------------------------------------------------
from google.colab import files
uploaded = files.upload()
df = pd.read_csv("spam.csv", encoding="latin-1")

# Keep only the required columns
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

print("First five records:")
print(df.head())

print("\nDataset shape:", df.shape)
print("\nClass distribution:")
print(df['label'].value_counts())

# --------------------------------------------------
# 2. Text Preprocessing
# --------------------------------------------------

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove punctuation and special characters
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Tokenization
    tokens = text.split()

    # Remove stopwords and apply stemming
    tokens = [
        stemmer.stem(word)
        for word in tokens
        if word not in stop_words
    ]

    return ' '.join(tokens)

df['cleaned_message'] = df['message'].apply(preprocess_text)

print("\nOriginal and preprocessed messages:")
print(df[['message', 'cleaned_message']].head())

# --------------------------------------------------
# 3. Feature Extraction using TF-IDF
# --------------------------------------------------

X = df['cleaned_message']
y = df['label']

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(X)

print("\nTF-IDF matrix shape:", X_tfidf.shape)

# --------------------------------------------------
# 4. Split Data into Training and Testing Sets
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

# --------------------------------------------------
# 5. Train Naive Bayes Classifier
# --------------------------------------------------

model = MultinomialNB()
model.fit(X_train, y_train)

print("\nNaive Bayes model trained successfully.")

# --------------------------------------------------
# 6. Make Predictions
# --------------------------------------------------

y_pred = model.predict(X_test)

print("\nPredictions:")
print(y_pred[:10])

# --------------------------------------------------
# 7. Evaluate Model
# --------------------------------------------------

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# --------------------------------------------------
# 8. Test with Custom Messages
# --------------------------------------------------

new_messages = [
    "Congratulations! You have won a free prize. Claim now!",
    "Hey, are we meeting for lunch today?",
    "URGENT! You have won a cash reward. Call now!"
]

# Apply the same preprocessing used for training
cleaned_messages = [
    preprocess_text(message) for message in new_messages
]

# Transform using the already-fitted TF-IDF vectorizer
new_messages_tfidf = tfidf.transform(cleaned_messages)

# Predict spam or ham
predictions = model.predict(new_messages_tfidf)

print("\nCustom Message Predictions:")
for message, prediction in zip(new_messages, predictions):
    print(f"Message: {message}")
    print(f"Prediction: {prediction.upper()}")
    print()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Saving spam.csv to spam.csv
First five records:
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

Dataset shape: (5572, 2)

Class distribution:
label
ham     4825
spam     747
Name: count, dtype: int64

Original and preprocessed messages:
                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                     cleaned_message  
0  go jurong point crazi avail bugi n great world...  
1        